In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import tqdm

In [ ]:
AGR = pd.read_csv('1600501_OIAPOQUE/Agregados_por_setores_demografia_BR.csv', sep = ';')
AGR

In [ ]:
# Sexo masculino, 0 a 4 anos     M0a
# Sexo masculino, 5 a 9 anos     M0b
# Sexo masculino, 10 a 14 anos   M1a
# Sexo masculino, 15 a 19 anos   M1b
# Sexo masculino, 20 a 24 anos   M2a
# Sexo masculino, 25 a 29 anos   M2b
# Sexo masculino, 30 a 39 anos   M3
# Sexo masculino, 40 a 49 anos   M4
# Sexo masculino, 50 a 59 anos   M5
# Sexo masculino, 60 a 69 anos   M6
# Sexo masculino, 70 anos ou mais M7
# Sexo feminino, 0 a 4 anos      F0a
# Sexo feminino, 5 a 9 anos      F0b
# Sexo feminino, 10 a 14 anos    F1a
# Sexo feminino, 15 a 19 anos    F1b
# Sexo feminino, 20 a 24 anos    F2a
# Sexo feminino, 25 a 29 anos    F2b
# Sexo feminino, 30 a 39 anos    F3
# Sexo feminino, 40 a 49 anos    F4
# Sexo feminino, 50 a 59 anos    F5
# Sexo feminino, 60 a 69 anos    F6
# Sexo feminino, 70 anos ou mais F7

#AGR_si = AGR[['COD_setor', 'V01009','V01010','V01011','V01012','V01013','V01014','V01015','V01016','V01017','V01018','V01019','V01020','V01021','V01022','V01023',
#     'V01024','V01025','V01026','V01027','V01028','V01029','V01030']]
dic_pop = {'V01006':'T','V01007':'M','V01008':'F','V01009':'M0a','V01010':'M0b','V01011':'M1a','V01012':'M1b',
    'V01013':'M2a', 'V01014':'M2b','V01015':'M3','V01016':'M4','V01017':'M5','V01018':'M6','V01019':'M7','V01020':'F0a','V01021':'F0b','V01022':'F1a',
    'V01023':'F1b','V01024':'F2a','V01025':'F2b','V01026':'F3','V01027':'F4','V01028':'F5','V01029':'F6','V01030':'F7',
    'V01031':'a0a','V01032':'a0b','V01033':'a1a','V01034':'a1b', 'V01035':'a2a','V01036':'a2b','V01037':'a3','V01038':'a4','V01039':'a5',
    'V01040':'a6','V01041':'a7'}
AGR_si = AGR.rename(columns = dic_pop)

AGR_si = AGR_si[AGR_si['T'] != 'X'].reset_index(drop=True)
AGR_si['M'] = AGR_si['M'].mask((AGR_si['M'] == 'X') & (AGR_si['F'] != 'X'), 
                               AGR_si['T'].astype('int') - AGR_si['F'].replace('X','0').astype('int'))
AGR_si['F'] = AGR_si['F'].mask((AGR_si['F'] == 'X') & (AGR_si['M'] != 'X'), 
                               AGR_si['T'].astype('int') - AGR_si['M'].replace('X','0').astype('int'))
for a in ['0a','0b','1a','1b','2a','2b','3','4','5','6','7']:
    for b in ['M','F']:
        c = 'M' if b == 'F' else 'F'
        AGR_si[f'{b}{a}'] = AGR_si[f'{b}{a}'].mask((AGR_si[f'{b}{a}'] == 'X') & 
                                                    (AGR_si[f'{c}{a}'] != 'X')&
                                                    (AGR_si[f'a{a}'] != 'X'),
                                                    AGR_si[f'a{a}'].replace('X','0').astype('int') - 
                                                    AGR_si[f'{c}{a}'].replace('X','0').astype('int'))
    AGR_si[f'a{a}'] = AGR_si[f'a{a}'].mask((AGR_si[f'a{a}'] == 'X') & 
                                                    (AGR_si[f'M{a}'] != 'X')&
                                                    (AGR_si[f'F{a}'] != 'X'),
                                                    AGR_si[f'M{a}'].replace('X','0').astype('int') +
                                                    AGR_si[f'F{a}'].replace('X','0').astype('int'))

In [ ]:
for idx in tqdm.tqdm(AGR_si[AGR_si.isin(['X']).any(axis=1)].index):
    s = AGR_si.loc[[idx],:]

    colunas_com_X = list(s.columns[s.eq('X').any()])
    colunas_com_X

    s = s.replace('X','2')
    s.loc[:,list(dic_pop.values())] = s.loc[:,list(dic_pop.values())].astype('int')
    a_s = [a for a in colunas_com_X if a[0] == 'a']
    a_s = list(np.random.choice(a_s, size = len(a_s), replace = False))

    s_ = s.iloc[0,:]

    for a in a_s:
        if s_.a0a+s_.a0b+s_.a1a+s_.a1b+s_.a2a+s_.a2b+s_.a3+s_.a4+s_.a5+s_.a6+s_.a7 > s_['T']:
            s_[a] = 1
            AGR_si.loc[idx, a] = '1'

    a_s = [a for a in colunas_com_X if a[0] != 'a']
    a_s = list(np.random.choice(a_s, size = len(a_s), replace = False))    

    a_s = [a for a in colunas_com_X if a[0] != 'a']
    a_s = list(np.random.choice(a_s, size = len(a_s), replace = False))

    a_s_copy = a_s.copy()

    for a in a_s_copy:
        comp =  'M'+a[1:] if a[0] == 'F' else 'F'+a[1:]
        if s_[a]+s_[comp] == s_['a'+a[1:]]:
            a_s.remove(a)
        elif comp not in a_s:
            s_[a] = 1
            AGR_si.loc[idx, a] = '1'
            a_s.remove(a)
        else:
            if a[0] == 'M':
                if s_.M0a+s_.M0b+s_.M1a+s_.M1b+s_.M2a+s_.M2b+s_.M3+s_.M4+s_.M5+s_.M6+s_.M7 > s_['M']:
                    s_[a] = 1
                    AGR_si.loc[idx, a] = '1'
                a_s.remove(a)
            elif a[0] == 'F':
                if s_.F0a+s_.F0b+s_.F1a+s_.F1b+s_.F2a+s_.F2b+s_.F3+s_.F4+s_.F5+s_.F6+s_.F7 > s_['F']:
                    s_[a] = 1
                    AGR_si.loc[idx, a] = '1'
                a_s.remove(a)
            else:
                print(a)  

In [ ]:
AGR_si.replace('X', '2', inplace=True)
AGR_si.iloc[:,1:]= AGR_si.iloc[:,1:].astype('int32')
AGR_si['COD_setor'] = AGR_si['CD_setor'].astype('str')
AGR_si

In [ ]:
AGR_si.to_csv('1600501_OIAPOQUE/Agregados_por_setores_demografia_BR_preenchido.csv', index=False, sep=';')